In [0]:
-- Each species is a single CSV file uploaded directly to the volume.
-- We keep all columns exactly as Movebank provided them.
-- Column names have hyphens (e.g. location-lat) — that is normal for
-- Movebank data. We rename them cleanly in Silver Workbook 2B.

In [0]:
-- Reef Manta Ray — Indonesia, 25 animals, 2014-2022

CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_manta_ray_raw AS
SELECT *,
current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/ReefMantaRay.csv',
format => 'csv',
header => true);

In [0]:
-- Whale Shark — Gulf of Mexico, 41 animals, 2009-2015


CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_whale_shark_raw AS
SELECT *,
current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/WhaleShark.csv',
format => 'csv',
header => true);

In [0]:
-- Sperm Whale — Gulf of Mexico, 17 animals, 2011-2013


CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_sperm_whale_raw AS
SELECT *,
current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/SpermWhale.csv',
format => 'csv',
header => true);

In [0]:
-- Green Turtle — Western Indian Ocean, 33 animals, 2012-2019


CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_green_turtle_raw AS
SELECT *,
current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/GreenTurtle.csv',
format => 'csv',
header => true);

In [0]:
-- VALIDATION — expected row counts:
-- Reef Manta Ray ~3,224 | Whale Shark ~3,382
-- Sperm Whale ~7,759    | Green Turtle ~55,411


SELECT 'Reef Manta Ray' AS species, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_manta_ray_raw UNION ALL
SELECT 'Whale Shark' AS species, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_whale_shark_raw UNION ALL
SELECT 'Sperm Whale' AS species, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw UNION ALL
SELECT 'Green Turtle' AS species, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_green_turtle_raw;

In [0]:
-- MICROPLASTICS
-- 471 plastic concentration observations from NOAA NCEI.
-- Atlantic and Caribbean region only — Indo-Pacific coverage is sparse.
-- Date range: 2010-2015. Units: pieces per cubic metre (pieces/m3).

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_microplastics_raw AS
SELECT 
  OBJECTID,
  `Latitude (degree)` AS latitude_degree,
  `Longitude (degree)` AS longitude_degree,
  Ocean,
  Region,
  Subregion,
  Country,
  State,
  `Beach Location` AS beach_location,
  `Marine Setting` AS marine_setting,
  `Ocean Bottom Depth (m)` AS ocean_bottom_depth_m,
  `Water Sample Depth (m)` AS water_sample_depth_m,
  `Sediment Sample Depth (m)` AS sediment_sample_depth_m,
  `Sampling Method` AS sampling_method,
  `Mesh Size (mm)` AS mesh_size_mm,
  `Transect Number` AS transect_number,
  `Sampling Point on Beach` AS sampling_point_on_beach,
  `Volunteers Number` AS volunteers_number,
  `Collecting Time (min)` AS collecting_time_min,
  `Standardized Nurdle  Amount` AS standardized_nurdle_amount,
  `Microplastics Measurement` AS microplastics_measurement,
  Unit,
  `Concentration Class Range` AS concentration_class_range,
  `Concentration Class` AS concentration_class,
  `Short Reference` AS short_reference,
  `Long Reference` AS long_reference,
  DOI,
  Organization,
  Keywords,
  `NCEI Accession Number` AS ncei_accession_number,
  `NCEI Accession Link` AS ncei_accession_link,
  Symbology,
  GlobalID,
  `Sample Date` AS sample_date,
  x,
  y,
  _rescued_data,
  current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/Marine_Microplastics.csv',
  format => 'csv',
  header => true);

In [0]:
-- Expected: 471 rows
SELECT count(*) AS total_rows 
FROM mcphersonsharyn_bronze.bronze_microplastics_raw;

-- Check what landed in _rescued_data
SELECT count(*) AS rows_with_rescued_data
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE _rescued_data IS NOT NULL;

In [0]:
-- VALIDATION — expected: 471 rows

SELECT count(*) AS total_rows FROM mcphersonsharyn_bronze.bronze_microplastics_raw;
 

In [0]:
-- See exactly what unit values exist

SELECT unit, count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
GROUP BY unit;

In [0]:

-- Check coordinate ranges

SELECT
round(min(CAST(latitude_degree AS DOUBLE)), 4) AS min_lat,
round(max(CAST(latitude_degree AS DOUBLE)), 4) AS max_lat,
round(min(CAST(longitude_degree AS DOUBLE)), 4) AS min_lon,
round(max(CAST(longitude_degree AS DOUBLE)), 4) AS max_lon
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3';

In [0]:
-- Check what the sample_date values actually look like
SELECT sample_date, count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3'
GROUP BY sample_date
LIMIT 10;

In [0]:

-- Test 1: Just the unit filter — should return 470
SELECT count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3';


In [0]:
-- Test 2: Add coordinate filters — should still return 470
SELECT count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3'
AND CAST(latitude_degree AS DOUBLE) BETWEEN -23 AND 22
AND CAST(longitude_degree AS DOUBLE) BETWEEN -73 AND 126;

In [0]:
-- Test 3: Add concentration filter
SELECT count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3'
AND CAST(latitude_degree AS DOUBLE) BETWEEN -23 AND 22
AND CAST(longitude_degree AS DOUBLE) BETWEEN -73 AND 126
AND CAST(microplastics_measurement AS DOUBLE) >= 0;

In [0]:
-- Test 4: Add date filter
SELECT count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3'
AND CAST(latitude_degree AS DOUBLE) BETWEEN -23 AND 22
AND CAST(longitude_degree AS DOUBLE) BETWEEN -73 AND 126
AND CAST(microplastics_measurement AS DOUBLE) >= 0
AND sample_date IS NOT NULL;

In [0]:

-- ENSO CLIMATE INDEX
-- Monthly El Nino / La Nina index (MEI.v2) from NOAA.
-- Covers 1979 to 2026 — every year in the ATLAS study period.

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_enso_raw AS
SELECT 
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[0] AS INT) AS year,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[1] AS DOUBLE) AS jan,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[2] AS DOUBLE) AS feb,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[3] AS DOUBLE) AS mar,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[4] AS DOUBLE) AS apr,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[5] AS DOUBLE) AS may,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[6] AS DOUBLE) AS jun,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[7] AS DOUBLE) AS jul,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[8] AS DOUBLE) AS aug,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[9] AS DOUBLE) AS sep,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[10] AS DOUBLE) AS oct,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[11] AS DOUBLE) AS nov,
  TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[12] AS DOUBLE) AS dec,
  current_timestamp() AS ingested_timestamp
FROM read_files('/Volumes/workspace/raw/environmental_data/Enso_Index.csv',
  format => 'csv',
  header => true)
WHERE TRY_CAST(SPLIT(TRIM(`1979     2026`), '\\s+')[0] AS INT) IS NOT NULL;

In [0]:
-- VALIDATION

SELECT count(*) AS total_rows FROM mcphersonsharyn_bronze.bronze_enso_raw;

In [0]:
SELECT 'CC master' AS table_name, count(*) AS rows, count(DISTINCT source_year) AS years FROM mcphersonsharyn_bronze.bronze_cc_master UNION ALL
SELECT 'SST master' AS table_name, count(*) AS rows, count(DISTINCT source_year) AS years FROM mcphersonsharyn_bronze.bronze_sst_master UNION ALL
SELECT 'Manta Ray' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_manta_ray_raw UNION ALL
SELECT 'Whale Shark' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_whale_shark_raw UNION ALL
SELECT 'Sperm Whale' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw UNION ALL
SELECT 'Green Turtle' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_green_turtle_raw UNION ALL
SELECT 'Microplastics' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_microplastics_raw UNION ALL
SELECT 'ENSO' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_enso_raw;

In [0]:
-- All tables showing rows → Bronze complete
-- SST 2022 is present in the SST master before proceeding
